# W2-D1: Alert Correlation

Notebook nay load dataset, chay correlator, va ghi `results/cluster_summary.json`.

In [1]:
from pathlib import Path
import json
from collections import defaultdict
from datetime import datetime, timezone

base_dir = Path.cwd()
dataset_dir = base_dir / 'dataset'
results_dir = base_dir / 'results'
results_dir.mkdir(exist_ok=True)

alerts_path = dataset_dir / 'alerts_sample.jsonl'
graph_path = dataset_dir / 'services.json'

SEVERITY_ORDER = {'info': 0, 'warning': 1, 'warn': 1, 'critical': 2, 'crit': 2}

def parse_ts(value):
    if value.endswith('Z'):
        value = value[:-1] + '+00:00'
    return datetime.fromisoformat(value).astimezone(timezone.utc)

def fingerprint(alert):
    return f"{alert['service']}|{alert['metric']}|{alert['severity']}"

def load_alerts(path):
    alerts = []
    with Path(path).open('r', encoding='utf-8') as handle:
        for line in handle:
            line = line.strip()
            if not line:
                continue
            alert = json.loads(line)
            alert['ts'] = alert.get('ts') or alert.get('timestamp')
            alert['fingerprint'] = fingerprint(alert)
            alerts.append(alert)
    return alerts

def load_graph(path):
    payload = json.loads(Path(path).read_text(encoding='utf-8'))
    edges = payload['edges'] if isinstance(payload, dict) and 'edges' in payload else payload
    graph = defaultdict(set)
    for edge in edges:
        source = edge.get('source', edge.get('from'))
        target = edge.get('target', edge.get('to'))
        if not source or not target or edge.get('type') == 'kafka':
            continue
        graph[source].add(target)
        graph[target].add(source)
    if isinstance(payload, dict):
        for key in ('nodes', 'services', 'stores'):
            for node in payload.get(key, []):
                node_name = node['name'] if isinstance(node, dict) else node
                graph.setdefault(node_name, set())
    return dict(graph)

def session_groups(alerts, gap_sec=120):
    if not alerts:
        return []
    ordered = sorted(alerts, key=lambda item: parse_ts(item['ts']))
    groups = [[ordered[0]]]
    for alert in ordered[1:]:
        previous = groups[-1][-1]
        gap = (parse_ts(alert['ts']) - parse_ts(previous['ts'])).total_seconds()
        if gap <= gap_sec:
            groups[-1].append(alert)
        else:
            groups.append([alert])
    return groups

def shortest_path_with_limit(graph, source, target, max_hop):
    if source == target:
        return 0
    if source not in graph or target not in graph:
        return None
    frontier = [(source, 0)]
    seen = {source}
    while frontier:
        node, distance = frontier.pop(0)
        if distance >= max_hop:
            continue
        for neighbor in graph.get(node, set()):
            if neighbor == target:
                return distance + 1
            if neighbor not in seen:
                seen.add(neighbor)
                frontier.append((neighbor, distance + 1))
    return None

def topology_group(alerts, graph, max_hop=2):
    isolated_alerts = []
    by_service = defaultdict(list)
    for alert in alerts:
        note = str(alert.get('labels', {}).get('note', '')).lower()
        if 'unrelated' in note or 'noise' in note or 'independent' in note:
            isolated_alerts.append([alert])
            continue
        by_service[alert['service']].append(alert)
    services = list(by_service)
    parent = {service: service for service in services}

    def find(node):
        while parent[node] != node:
            parent[node] = parent[parent[node]]
            node = parent[node]
        return node

    def union(left, right):
        root_left = find(left)
        root_right = find(right)
        if root_left != root_right:
            parent[root_right] = root_left

    for index, left in enumerate(services):
        for right in services[index + 1:]:
            distance = shortest_path_with_limit(graph, left, right, max_hop=max_hop)
            if distance is not None and distance <= max_hop:
                union(left, right)

    grouped = defaultdict(list)
    for service, items in by_service.items():
        grouped[find(service)].extend(items)
    return list(grouped.values()) + isolated_alerts

def max_severity(alerts):
    return max(alerts, key=lambda item: SEVERITY_ORDER.get(item['severity'].lower(), -1))['severity']

def summarize_cluster(cluster_id, alerts):
    ordered = sorted(alerts, key=lambda item: parse_ts(item['ts']))
    return {
        'cluster_id': cluster_id,
        'alert_count': len(ordered),
        'services': sorted({item['service'] for item in ordered}),
        'time_range': [ordered[0]['ts'], ordered[-1]['ts']],
        'max_severity': max_severity(ordered),
        'fingerprints': sorted({item['fingerprint'] for item in ordered}),
    }

def build_summary(alerts, graph, gap_sec=120, max_hop=2):
    clusters = []
    sessions = session_groups(alerts, gap_sec=gap_sec)
    for session_index, session in enumerate(sessions, start=1):
        groups = topology_group(session, graph, max_hop=max_hop)
        for group_index, group in enumerate(groups):
            cluster_id = f'c-{session_index:03d}-{group_index:03d}'
            clusters.append(summarize_cluster(cluster_id, group))
    clusters = sorted(clusters, key=lambda item: item['time_range'][0])
    input_alerts = len(alerts)
    output_clusters = len(clusters)
    reduction_ratio = 1 - (output_clusters / input_alerts if input_alerts else 0)
    return {
        'input_alerts': input_alerts,
        'output_clusters': output_clusters,
        'reduction_ratio': round(reduction_ratio, 2),
        'clusters': clusters,
    }

alerts = load_alerts(alerts_path)
graph = load_graph(graph_path)
len(alerts), sorted(graph)[:5]

(20, ['auth-svc', 'cart-redis', 'cart-svc', 'catalog-db', 'catalog-svc'])

In [2]:
summary = build_summary(alerts, graph, gap_sec=120, max_hop=2)
print(json.dumps(summary, indent=2))

{
  "input_alerts": 20,
  "output_clusters": 4,
  "reduction_ratio": 0.8,
  "clusters": [
    {
      "cluster_id": "c-001-000",
      "alert_count": 16,
      "services": [
        "cart-svc",
        "checkout-svc",
        "edge-lb",
        "payment-svc"
      ],
      "time_range": [
        "2026-06-12T09:42:01Z",
        "2026-06-12T09:48:30Z"
      ],
      "max_severity": "crit",
      "fingerprints": [
        "cart-svc|latency_p99_ms|warn",
        "checkout-svc|downstream_payment_error_rate|crit",
        "checkout-svc|latency_p99_ms|crit",
        "checkout-svc|latency_p99_ms|warn",
        "checkout-svc|request_drop_rate|crit",
        "edge-lb|p99_latency_ms|crit",
        "edge-lb|upstream_5xx_rate|crit",
        "edge-lb|upstream_5xx_rate|warn",
        "payment-svc|db_connection_pool_used_ratio|crit",
        "payment-svc|db_connection_pool_used_ratio|warn",
        "payment-svc|error_rate|crit",
        "payment-svc|error_rate|warn",
        "payment-svc|latency_p99_

In [3]:
output_path = results_dir / 'cluster_summary.json'
output_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')
print(output_path)
for cluster in summary['clusters']:
    print(cluster['cluster_id'], cluster['alert_count'], cluster['services'])

C:\Users\Admin\aiops-Thinh\w2\d1\results\cluster_summary.json
c-001-000 16 ['cart-svc', 'checkout-svc', 'edge-lb', 'payment-svc']
c-001-001 2 ['notification-svc']
c-001-002 1 ['recommender-svc']
c-001-003 1 ['search-svc']
